# 노션 트러블슈팅 검색기


**프로젝트 개요**

기존 노션 검색의 한계인 제목 기반 검색을 벗어나, 문서 내용까지 의미적으로 검색할 수 있는 시스템을 구현했습니다.
```

노션 DB Export (Markdown)
        ↓
문서 청킹 (RecursiveCharacterTextSplitter)
        ↓
임베딩 변환 (OpenAI Embeddings)
        ↓
벡터 DB 저장 (Chroma)
        ↓
의미 기반 검색
        ↓
관련 문서 반환
```


In [1]:
!pip3 install -qU langchain-chroma
!pip3 install -qU langchain-openai
!pip3 install -qU langchain-text-splitters
!pip3 install -qU gradio python-dotenv
!pip3 install -qU "huggingface_hub==0.34.4"


You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


### API 키 설정

In [2]:
from pathlib import Path
from datetime import datetime
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
from dotenv import load_dotenv
import gradio as gr
import os
import json

load_dotenv()
print("OPENAI_API_KEY loaded:", os.getenv("OPENAI_API_KEY") is not None)

# API 키 잘 로드됐는지 확인
print(os.environ.get("OPENAI_API_KEY")[:5] + "*****")

embeddings = OpenAIEmbeddings()
print("✅ 임베딩 모델 준비 완료!")


/Users/eunchae/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/eunchae/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


OPENAI_API_KEY loaded: True
sk-pr*****
✅ 임베딩 모델 준비 완료!


### 노션에서 Export 한 zip 파일 압축 해제

In [3]:
import zipfile
import os
from pathlib import Path

if Path("./03_RAG/notion_troubleshooting_search/dummy_data.zip").exists():
    BASE_DIR = Path("./03_RAG/notion_troubleshooting_search").resolve()
else:
    BASE_DIR = Path(".").resolve()

print(f"📁 BASE_DIR: {BASE_DIR}")

ZIP_PATH = BASE_DIR / "dummy_data.zip"
DATA_DIR = BASE_DIR / "dummy_data"
CHROMA_DIR = BASE_DIR / "notion_chroma"

if not DATA_DIR.exists():
    with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
        for member in zip_ref.infolist():
            name = member.filename
            try:
                name = name.encode("cp437").decode("utf-8")
            except Exception:
                pass

            target_path = BASE_DIR / name
            if member.is_dir():
                target_path.mkdir(parents=True, exist_ok=True)
                continue

            target_path.parent.mkdir(parents=True, exist_ok=True)
            with zip_ref.open(member) as src, open(target_path, "wb") as dst:
                dst.write(src.read())
    print("✅ 압축 해제 완료!")
else:
    print("✅ 이미 압축 해제됨, 스킵!")

# 파일 목록 확인
#for root, dirs, files in os.walk("./dummy_data"):
#    for file in files:
#        print(file)


📁 BASE_DIR: /Users/eunchae/repository/skax/03_RAG/notion_troubleshooting_search
✅ 이미 압축 해제됨, 스킵!


In [4]:
# ============================================
# md 파일 경로 수집
# ============================================
import glob

md_paths = glob.glob(str(DATA_DIR / "**/*.md"), recursive=True)
md_paths = [p for p in md_paths if not p.endswith("SUMMARY.md")]
print(f"📄 총 {len(md_paths)}개 md 파일 발견!")

📄 총 10개 md 파일 발견!


In [6]:
import unicodedata

# ============================================
# 문서 읽기
# ============================================
docs = [] # LangChain Document 객체 리스트. 벡터 DB 저장용
original_docs = {} # 내용 전체
doc_meta = {}

for full_path in md_paths:
    with open(full_path, "r", encoding="utf-8") as f:
        content = f.read().strip()
        if not content:
            continue

        path_obj = Path(full_path)
        title = unicodedata.normalize("NFC", path_obj.stem)
        stat = path_obj.stat()
        created_ts = getattr(stat, "st_birthtime", stat.st_mtime)
        created_at = datetime.fromtimestamp(created_ts).strftime("%Y-%m-%d")

        original_docs[title] = content
        doc_meta[title] = {
            "created_at": created_at,
            "created_ts": created_ts,
            "full_path": full_path,
        }
        docs.append(Document(
            page_content=content,
            metadata={
                "source": f"{title}.md",
                "full_path": full_path,
                "created_at": created_at,
            },
        ))

print(f"✅ 원본 문서 {len(original_docs)}개 준비 완료")
print("\n파일 목록:")
for path in md_paths[:5]:
    print(" -", path)

✅ 원본 문서 10개 준비 완료

파일 목록:
 - /Users/eunchae/repository/skax/03_RAG/notion_troubleshooting_search/dummy_data/troubleshooting/MySQL_ONLY_FULL_GROUP_BY_오류.md
 - /Users/eunchae/repository/skax/03_RAG/notion_troubleshooting_search/dummy_data/troubleshooting/Spring_의존성_충돌_오류.md
 - /Users/eunchae/repository/skax/03_RAG/notion_troubleshooting_search/dummy_data/troubleshooting/Swagger_array_타입_오류.md
 - /Users/eunchae/repository/skax/03_RAG/notion_troubleshooting_search/dummy_data/troubleshooting/PR_서버_적용_오류.md
 - /Users/eunchae/repository/skax/03_RAG/notion_troubleshooting_search/dummy_data/troubleshooting/Elasticsearch_실행_안됨.md


### 벡터 DB 설정

In [7]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
)


In [8]:

import shutil

#DB 업데이트 여부 설정 
FORCE_UPDATE = True  # 처음엔 True로!

In [9]:

CHROMA_DIR.parent.mkdir(parents=True, exist_ok=True)
print(f"📁 CHROMA_DIR: {CHROMA_DIR}")

if not CHROMA_DIR.exists() or FORCE_UPDATE:
    # 기존 DB 삭제
    if CHROMA_DIR.exists():
        shutil.rmtree(CHROMA_DIR)
        print("🗑️ 기존 벡터 DB 삭제!")

    #새로만들기
    chunks = text_splitter.split_documents(docs)
    print(f"✂️ 총 {len(chunks)}개 청크!")

    CHROMA_DIR.mkdir(parents=True, exist_ok=True)

    vector_store = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        collection_name="troubleshooting_db",
        persist_directory=str(CHROMA_DIR)
    )
    print(f"✅ 저장 완료! {vector_store._collection.count()}개")
else:
    # 기존 DB 로드
    vector_store = Chroma(
        collection_name="troubleshooting_db",
        embedding_function=embeddings,
        persist_directory=str(CHROMA_DIR)
    )
    print(f"✅ 기존 DB 로드! {vector_store._collection.count()}개")

    

📁 CHROMA_DIR: /Users/eunchae/repository/skax/03_RAG/notion_troubleshooting_search/notion_chroma
🗑️ 기존 벡터 DB 삭제!
✂️ 총 20개 청크!
✅ 저장 완료! 20개


### 프롬프트

In [10]:
query_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 요약용
summary_llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
)

# 해결책 제안 및 Q&A용
suggest_llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.7,
)

query_prompt = ChatPromptTemplate.from_template("""
너는 트러블슈팅 검색을 위한 질의 정규화기다.

목표:
- 사용자 질문에서 핵심 기술 키워드를 추출
- 한글/영문/대소문자가 다른 동일 개념을 같은 의미로 처리
- 검색에 유리하도록 대표 검색어와 확장 검색어를 생성

동의어 규칙 예시:
- jenkins = Jenkins = 젠킨스
- github = GitHub = 깃허브
- swagger = Swagger = 스웨거
- spring boot = Spring Boot = 스프링부트
- mysql = MySQL = 마이에스큐엘
- tls = TLS = ssl = SSL

지침:
- 같은 의미의 표현은 하나의 개념으로 묶기
- 원문 의미를 바꾸지 말기
- 검색에 불필요한 조사나 장식어는 제거 가능
- 출력은 반드시 JSON 형태로만 반환

출력 형식:
{{
  "normalized_query": "대표 검색어",
  "expanded_keywords": ["동의어1", "동의어2", "동의어3"],
  "intent": "사용자가 찾고 싶은 문제 상황 요약"
}}

사용자 질문:
{question}
""")

print("✅ 프롬프트 정의 완료!")




✅ 프롬프트 정의 완료!


### 쿼리 정규화 함수

In [11]:


query_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def normalize_query(question: str) -> dict:
    """사용자 질문을 정규화하여 검색어 확장"""
    fallback = {
        "normalized_query": question,
        "expanded_keywords": [],
        "intent": question,
    }

    try:
        response = query_llm.invoke(query_prompt.format(question=question))
        content = (response.content or "").strip()
        if not content:
            return fallback

        try:
            parsed = json.loads(content)
        except json.JSONDecodeError:
            start = content.find("{")
            end = content.rfind("}")
            if start == -1 or end == -1 or start >= end:
                return fallback
            parsed = json.loads(content[start:end + 1])

        return {
            "normalized_query": parsed.get("normalized_query") or question,
            "expanded_keywords": parsed.get("expanded_keywords") or [],
            "intent": parsed.get("intent") or question,
        }
    except Exception:
        return fallback

def dedupe_by_title(documents: list) -> list:
    """제목 기준 중복 문서 제거"""
    seen = set()
    unique_docs = []
    for doc in documents:
        title = doc.metadata.get("source", "unknown").replace(".md", "")
        if title not in seen:
            seen.add(title)
            unique_docs.append(doc)
    return unique_docs

def normalize_search_text(text: str) -> str:
    text = (text or "").lower()
    for ch in ["_", "-", "/", "(", ")", ".", ",", ":", "?", "!", "\n"]:
        text = text.replace(ch, " ")
    return " ".join(text.split())

def extract_keywords(*values: str) -> list:
    keywords = []
    for value in values:
        normalized = normalize_search_text(value)
        if not normalized:
            continue
        keywords.append(normalized)
        for token in normalized.split():
            if len(token) >= 2:
                keywords.append(token)
    return list(dict.fromkeys(keywords))

def score_keyword_match(title: str, content: str, keywords: list) -> int:
    title_text = normalize_search_text(title)
    content_text = normalize_search_text(content)
    score = 0

    for keyword in keywords:
        keyword_text = normalize_search_text(keyword)
        if not keyword_text:
            continue
        if keyword_text == title_text:
            score += 40
        elif keyword_text in title_text:
            score += 20
        if keyword_text in content_text:
            score += 6
    return score

def make_doc_from_store(title: str, content: str) -> Document:
    return Document(
        page_content=content,
        metadata={
            "source": f"{title}.md",
            "full_path": doc_meta[title]["full_path"],
            "created_at": doc_meta[title]["created_at"],
        },
    )

def search_documents(vector_store, question: str, k: int = 8, mode: str = "search") -> dict:
    """검색창은 키워드 중심, Q&A는 하이브리드 검색"""
    question = (question or "").strip()
    if not question:
        return {
            "query_info": {
                "normalized_query": "",
                "expanded_keywords": [],
                "intent": "",
            },
            "documents": [],
        }

    if mode == "qa":
        query_info = normalize_query(question)
        queries = [question, query_info["normalized_query"]] + query_info["expanded_keywords"]
        keywords = extract_keywords(question, query_info["normalized_query"], *query_info["expanded_keywords"])
    else:
        query_info = {
            "normalized_query": question,
            "expanded_keywords": [],
            "intent": question,
        }
        queries = [question]
        keywords = extract_keywords(question)

    lexical_matches = []
    for title, content in original_docs.items():
        lexical_score = score_keyword_match(title, content, keywords)
        if lexical_score > 0:
            lexical_matches.append({
                "score": lexical_score,
                "doc": make_doc_from_store(title, content),
            })

    lexical_matches.sort(key=lambda item: item["score"], reverse=True)

    if mode == "search" and lexical_matches:
        return {
            "query_info": query_info,
            "documents": [item["doc"] for item in lexical_matches[:k]],
        }

    scored_by_title = {}
    for item in lexical_matches:
        title = item["doc"].metadata.get("source", "unknown").replace(".md", "")
        scored_by_title[title] = item

    candidate_k = max(k * 2, 8)
    for q in list(dict.fromkeys([query for query in queries if query])):
        docs = vector_store.similarity_search(q, k=candidate_k)
        for rank, doc in enumerate(docs):
            title = doc.metadata.get("source", "unknown").replace(".md", "")
            lexical_score = score_keyword_match(title, doc.page_content, keywords)
            if mode == "search" and lexical_score <= 0:
                continue
            vector_score = max(candidate_k - rank, 1)
            total_score = lexical_score + vector_score

            existing = scored_by_title.get(title)
            if existing is None or total_score > existing["score"]:
                scored_by_title[title] = {
                    "score": total_score,
                    "doc": doc,
                }

    ranked_docs = [
        item["doc"]
        for item in sorted(
            scored_by_title.values(),
            key=lambda item: item["score"],
            reverse=True,
        )
    ]

    return {
        "query_info": query_info,
        "documents": dedupe_by_title(ranked_docs)[:k],
    }

print("✅ 검색 함수 정의 완료!")

✅ 검색 함수 정의 완료!


### Gradio 함수

In [ ]:
# ============================================
# Gradio 함수
# ============================================

def make_item(title, mode="all"):
    """문서 아이템 생성"""
    if mode == "all":
        subtitle = f"🗓️ {doc_meta[title]['created_at']}"
    else:
        snippet = original_docs.get(title, "").replace("\n", " ").strip()[:140]
        subtitle = f"📝 {snippet}..."
    
    return {
        "title": title,
        "mode": mode,
        "subtitle": subtitle  # ← 항상 subtitle 키 포함!
    }

def build_items(query: str) -> list:
    """검색어 유무에 따라 문서 목록 생성"""
    query = (query or "").strip()
    if not query:
        # 검색어 없으면 전체 문서 최신순
        titles = sorted(
            original_docs.keys(),  # ← docs → original_docs!
            key=lambda x: doc_meta[x]["created_ts"],
            reverse=True,
        )
        return [make_item(title, mode="all") for title in titles]

    # 검색어 있으면 검색 결과
    result = search_documents(vector_store, query, k=10, mode="search")
    titles = [
        doc.metadata["source"].replace(".md", "")
        for doc in result["documents"]
    ]
    return [make_item(title, mode="search") for title in titles]
    
def search_and_update(query: str):
    """검색 실행 후 문서 목록 갱신"""
    items = build_items(query)
    titles = [item["title"] for item in items]
    return gr.update(choices=titles, value=None), "문서 제목을 클릭하면 전체 내용을 볼 수 있어요."

def search(query: str):
    return search_and_update(query)

def show_full_doc(title: str):
    if not title:
        return "문서를 찾을 수 없어요"
    content = original_docs.get(title, "문서를 찾을 수 없어요")

    # return f"## 📄 {title}\n\n---\n\n{content}"
    content = f"## 📄 {title}\n\n---\n\n{content}"
    return content

print("✅ Gradio 함수 정의 완료!")

In [ ]:
# ============================================
# Gradio 함수 - 문서 요약
# ============================================
def summarize_doc(title: str) -> str:
    """문서 내용 AI 요약"""
    if not title:
        return "문서를 먼저 선택해주세요!"
    
    content = original_docs.get(title, "")
    if not content:
        return "문서 내용을 찾을 수 없어요"
    
    prompt = f"""
다음 트러블슈팅 문서를 아래 형식으로 요약해줘.

## 🔴 문제 상황
(어떤 문제가 발생했는지)

## 🔍 원인
(왜 발생했는지)

## ✅ 해결 방법
(어떻게 해결했는지)

문서 내용:
{content[:3000]}
"""
    return summary_llm.invoke(prompt).content


# 스트리밍 함수
def summarize_stream(title: str):
    if not title:
        yield gr.update(value="문서를 먼저 선택해주세요!", visible=True)
        return
    
    content = original_docs.get(title, "")
    if not content:
        yield gr.update(value="문서 내용을 찾을 수 없어요", visible=True)
        return
    
    prompt = f"""
다음 트러블슈팅 문서를 아래 형식으로 요약해줘.

🔴 문제: (한 줄)
🔍 원인: (한 줄)
✅ 해결: (한 줄)

문서:
{content[:3000]}
"""
    # 스트리밍으로 토큰 하나씩 반환!
    result = ""
    for chunk in summary_llm.stream(prompt):
        result += chunk.content
        yield gr.update(value=result, visible=True)

def ask_and_search(question: str):
    """질문과 함께 관련 문서를 검색하고 답변 스트리밍"""
    question = (question or "").strip()
    if not question:
        yield (
            gr.update(choices=[item["title"] for item in build_items("")], value=None),
            "질문을 입력해주세요.",
            gr.update(visible=False),
            gr.update(value="", visible=False),
        )
        return

    result = search_documents(vector_store, question, k=5, mode="qa")
    docs = result["documents"]
    if not docs:
        yield (
            gr.update(choices=[], value=None),
            "관련 문서가 없어요",
            gr.update(visible=False),
            gr.update(value="", visible=False),
        )
        return

    titles = [doc.metadata["source"].replace(".md", "") for doc in docs]
    context = "\n\n".join([doc.page_content for doc in docs])
    prompt = ChatPromptTemplate.from_messages([
        (
            "system",
            """
당신은 트러블슈팅 Q&A 시스템입니다.

[역할]
- 개발자의 트러블슈팅 질문에 답변
- 반드시 제공된 문서를 기반으로만 답변
- 문서에 없는 내용은 절대 지어내지 않음

[답변 형식]
## 🔴 문제 상황
(질문에서 파악한 문제 요약)

## ✅ 해결 방법
(문서 기반 해결책, 단계별 번호로)

## 📄 참고 문서
(참고한 문서 제목 목록)

## ⚠️ 주의사항
(있을 경우에만 작성)

[규칙]
- 문서에 없으면 '관련 문서가 없어요' 라고 명확히 답변
- 코드/설정값은 코드블록으로 표시
- 한국어로 친근하게 답변
- 문서에 없는 내용 절대 지어내지 않음

[참고 문서]
{context}
""",
        ),
        ("user", "{question}"),
    ])

    parser = StrOutputParser()
    messages = prompt.format_messages(context=context, question=question)
    answer = ""
    yield (
        gr.update(choices=titles, value=None),
        "",
        gr.update(visible=False),
        gr.update(value="", visible=False),
    )
    for chunk in suggest_llm.stream(messages):
        text = parser.parse(chunk.content) if chunk.content else ""
        answer += text
        yield (
            gr.update(choices=titles, value=None),
            answer,
            gr.update(visible=False),
            gr.update(value="", visible=False),
        )


### Gradio UI

In [ ]:
# Gradio UI 셀 전에 실행!

# 1. 데이터 확인
print("=== 데이터 확인 ===")
print(f"original_docs: {len(original_docs)}개")
print(f"doc_meta: {len(doc_meta)}개")

# 2. build_items 결과 확인
items = build_items("")
print(f"\n=== build_items 결과 ===")
print(f"총 {len(items)}개")
print(f"샘플: {items[:2]}")

# 3. make_item 직접 테스트
title = list(original_docs.keys())[0]
item = make_item(title, mode="all")
print(f"\n=== make_item 결과 ===")
print(item)

In [ ]:
# ============================================
# Gradio UI
# ============================================

custom_css = """
body {
  overflow: hidden !important;
}
.gradio-container {
  height: 100vh !important;
  overflow: hidden !important;
}
#doc-left-panel,
#doc-right-panel {
  display: flex !important;
  flex-direction: column !important;
  height: 720px;
  min-height: 0 !important;
}
#doc-left-panel {
  border-right: 1px solid #e5e7eb;
  padding-right: 12px;
  overflow: hidden;
}
#doc-right-panel {
  padding-left: 12px;
  flex-direction: column !important;
  overflow-y: auto;
}
#doc-content-stack {
  display: flex !important;
  flex-direction: column !important;
  gap: 12px !important;
  width: 100% !important;
  min-height: 0 !important;
  flex: 1 1 auto !important;
}
#doc-left-stack {
  display: flex !important;
  flex-direction: column !important;
  gap: 12px !important;
  width: 100% !important;
  height: 100% !important;
  min-height: 0 !important;
  flex: 1 1 auto !important;
}
#qa-wrap {
  flex: 0 0 auto !important;
}
#search-wrap {
  flex: 0 0 auto !important;
}
#doc-list-wrap {
  display: block !important;
  flex: 1 1 auto !important;
  min-height: 0 !important;
  max-height: 100% !important;
  overflow-y: auto !important;
}
.doc-card {
  border-bottom: 1px solid #e5e7eb;
  padding: 10px 0;
}
.doc-title-btn button {
  justify-content: flex-start !important;
  text-align: left !important;
  font-size: 1.05rem !important;
  font-weight: 700 !important;
  border: none !important;
  background: transparent !important;
  box-shadow: none !important;
  padding: 0 !important;
  color: #1f2937 !important;
}
.doc-date p {
  font-size: 0.76rem !important;
  color: #6b7280 !important;
  margin-top: 4px !important;
  margin-bottom: 0 !important;
}
.doc-preview p {
  font-size: 0.88rem !important;
  color: #6b7280 !important;
  margin-top: 4px !important;
  margin-bottom: 0 !important;
  line-height: 1.5 !important;
}
#doc-full-content {
  display: block;
  min-height: 100%;
}
#doc-title {
  display: block;
  padding-bottom: 8px;
  border-bottom: 2px solid #e5e7eb;
  margin-bottom: 12px;
}
#doc-title h2 {
  display: block;
  font-size: 1.2rem !important;
  font-weight: 700 !important;
  color: #ffffff !important;
}
#summary-callout {
  background-color: #eff6ff !important;
  border-left: 4px solid #3b82f6 !important;
  border-radius: 8px !important;
  padding: 16px !important;
  margin-bottom: 16px !important;
  width: 100% !important;
  box-sizing: border-box !important;
}
#summary-callout p {
  color: #1e40af !important;
  font-size: 0.95rem !important;
  line-height: 1.6 !important;
}

"""

with gr.Blocks(css=custom_css) as demo:
    gr.Markdown("# 🔍 노션 트러블슈팅 검색기")
    gr.Markdown("검색어가 없으면 전체 문서 목록, 검색어가 있으면 검색 결과 문서 목록을 표시")

    with gr.Row():
        with gr.Column(scale=4, elem_id="doc-left-panel"):
          with gr.Column(elem_id="doc-left-stack"):
              with gr.Column(elem_id="qa-wrap"):
                  gr.Markdown("### 💬 Q&A")
                  qa_input = gr.Textbox(
                      show_label=False,
                      lines=1,
                      placeholder="예: jenkins 웹훅 안될때 어떻게 해?",
                  )
                  qa_output = gr.Markdown(value="", elem_id="qa-output")

              gr.Markdown("---")

              with gr.Column(elem_id="search-wrap"):
                  search_input = gr.Textbox(
                      label="🔍 검색어",
                      placeholder="예: DB 오류, jenkins...",
                  )

              with gr.Group(elem_id="doc-list-wrap"):
                  doc_list = gr.Radio(
                      label="📋 문서 목록",
                      choices=[item["title"] for item in build_items("")],
                      value=None,
                      interactive=True,
                  )
        with gr.Column(scale=6, elem_id="doc-right-panel"):
            # with gr.Row():
            #   summarize_btn = gr.Button("✨ AI 요약", scale=1)
            with gr.Column(elem_id="doc-content-stack"):
              summarize_btn = gr.Button("✨ AI 요약")
              summary_box = gr.Markdown(
                value="",
                elem_id="summary-callout",
                visible=False,
              )
              doc_viewer = gr.Markdown(
                  value="문서 제목을 클릭하면 전체 내용을 볼 수 있어요.",
                  elem_id="doc-full-content"
              )


    search_input.submit(
        fn=search_and_update,
        inputs=search_input,
        outputs=[doc_list, doc_viewer],

    )

    qa_input.submit(
        fn=ask_and_search,
        inputs=qa_input,
        outputs=[doc_list, doc_viewer, summarize_btn, summary_box],
    )

    doc_list.change(
        fn=lambda title: (show_full_doc(title), gr.update(visible=True), gr.update(value="", visible=False)),
        inputs=doc_list,
        outputs=[doc_viewer, summarize_btn, summary_box]

    )

  
    # AI 요약 버튼 → 스트리밍으로 콜아웃에 표시
    summarize_btn.click(
        fn=summarize_stream,
        inputs=doc_list,
        outputs=summary_box  # ← 콜아웃에만!
    )


    demo.load(
        fn=lambda: search_and_update(""),
        # outputs=[doc_list, doc_title, doc_viewer],
        outputs=[doc_list, doc_viewer],

    )

demo.launch()
